# Kiểm tra backward, training CIFAR-10 nhỏ và nạp checkpoint
thực hiện các phép thử ngắn này và tạo checkpoint/logs trong `artifacts/notebook_smoke/cifar_subset/`. Accuracy ở đây chỉ phục vụ kiểm tra luồng code, chưa dùng để kết luận model nào tốt hơn. Không đánh giá model trên test set trong notebook này.

## 1. Môi trường và cấu hình

Các biến được tạo lại trong notebook này; notebook không phụ thuộc biến còn trong kernel của notebook 01/02. Đọc JSON của project rồi tạo **bản sao cấu hình smoke**: 224×224, batch32, worker0, một epoch, pretrained và fine-tune toàn bộ backbone.

Worker0 giảm chi phí khởi tạo tiến trình với tập con nhỏ. Cấu hình chính thức worker4 đã có bước kiểm tra DataLoader riêng.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import os
import sys
import io
import gc
import csv
import json
import math
import logging

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'src/cifar_compare/train.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở notebook từ thư mục project hoặc notebooks.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10
from src.cifar_compare.models import build_model
from src.cifar_compare.data import get_dataloaders
from src.cifar_compare.train import DEFAULTS, build_optimizer, train_model, run_epoch
from src.cifar_compare.utils import load_config, set_seed, resolve_device

base_config = load_config('configs/default.json', DEFAULTS)
smoke_config = dict(base_config)
smoke_config.update({'input_size': 224, 'batch_size': 32, 'num_workers': 0,
                     'epochs': 1, 'pretrained': True, 'freeze_backbone': False,
                     'early_stopping_patience': 0,
                     'output_dir': 'artifacts/notebook_smoke/cifar_subset'})
DEVICE = resolve_device(smoke_config['device'])
SEED = int(smoke_config['seed'])
RUN_GROUP = datetime.now(ZoneInfo('Asia/Singapore')).strftime('%Y%m%d_%H%M%S_%f')
check_dir = Path(smoke_config['output_dir']) / 'checks' / RUN_GROUP
check_dir.mkdir(parents=True, exist_ok=False)
checks = {}
training_results = {}
reload_results = {}
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s', force=True)
print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))
print('Run group:', RUN_GROUP)
print('Cấu hình smoke:', {k: smoke_config[k] for k in
                          ('input_size', 'batch_size', 'num_workers', 'epochs', 'pretrained', 'freeze_backbone')})

Python: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\.venv-1\Scripts\python.exe
PyTorch: 2.14.1+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti
Run group: 20261109_005858_994753
Cấu hình smoke: {'input_size': 224, 'batch_size': 32, 'num_workers': 0, 'epochs': 1, 'pretrained': True, 'freeze_backbone': False}


## A. Kiểm tra một bước backward của ConvNeXt-Tiny

- **Forward:** ảnh → 10 logits.
- **Loss:** CrossEntropy so sánh logits và nhãn.
- **Backward:** tính gradient, cho biết trọng số cần điều chỉnh thế nào.
- **Optimizer step:** cập nhật trọng số từ gradient.

Cell dùng hai tensor giả để kiểm tra cơ chế, với weights pretrained và `freeze_backbone=False`. Kiểm tra gradient hữu hạn của các tham số, gradient khác 0 ở một trọng số backbone và classifier, rồi xác nhận hai trọng số đó thay đổi sau `optimizer.step()`.

Phép thử đơn bước dùng float32 để dễ kiểm tra gradient. Phần B chạy training loop của project với AMP theo cấu hình.

In [2]:
def check_convnext_update():
    set_seed(SEED)
    model = build_model('convnext_tiny', num_classes=10, pretrained=True,
                        weights_name=smoke_config['weights'], freeze_backbone=False).to(DEVICE)
    model.train()
    optimizer = build_optimizer(model, smoke_config)
    backbone_name, backbone_weight = next(
        (n, p) for n, p in model.named_parameters()
        if not n.startswith('classifier.') and p.requires_grad and p.ndim >= 2
    )
    head_weight = model.classifier[-1].weight
    before_backbone = backbone_weight.detach().clone()
    before_head = head_weight.detach().clone()

    images = torch.randn(2, 3, smoke_config['input_size'], smoke_config['input_size'], device=DEVICE)
    labels = torch.tensor([0, 1], device=DEVICE, dtype=torch.int64)
    optimizer.zero_grad(set_to_none=True)
    logits = model(images)
    assert tuple(logits.shape) == (2, 10)
    assert torch.isfinite(logits).all().item()
    loss = F.cross_entropy(logits, labels)
    assert torch.isfinite(loss).item(), 'Loss chứa NaN/Inf.'
    loss.backward()

    gradients = [(n, p.grad) for n, p in model.named_parameters() if p.requires_grad]
    for name, gradient in gradients:
        assert gradient is not None, f'Tham số không nhận gradient: {name}'
        assert torch.isfinite(gradient).all().item(), f'Gradient NaN/Inf: {name}'
    head_grad_norm = head_weight.grad.norm().item()
    backbone_grad_norm = backbone_weight.grad.norm().item()
    assert head_grad_norm > 0 and backbone_grad_norm > 0, 'Gradient được kiểm tra phải khác 0.'
    optimizer.step()
    assert not torch.equal(before_head, head_weight.detach()), 'Classifier chưa cập nhật.'
    assert not torch.equal(before_backbone, backbone_weight.detach()), 'Backbone chưa cập nhật.'
    return {'loss': loss.item(), 'head_grad_norm': head_grad_norm,
            'backbone_grad_norm': backbone_grad_norm, 'backbone_parameter': backbone_name,
            'head_updated': True, 'backbone_updated': True}

checks['backward'] = False
backward_result = check_convnext_update()
checks['backward'] = True
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print(backward_result)
print('PASS — ConvNeXt-Tiny backward: gradient hợp lệ, backbone và classifier đã cập nhật.')

{'loss': 2.169595956802368, 'head_grad_norm': 7.719360828399658, 'backbone_grad_norm': 1.2312541007995605, 'backbone_parameter': 'features.0.0.weight', 'head_updated': True, 'backbone_updated': True}
PASS — ConvNeXt-Tiny backward: gradient hợp lệ, backbone và classifier đã cập nhật.


## B1. Chọn tập con CIFAR-10 thật

Mỗi lớp chọn **20 ảnh train và 10 ảnh validation** bằng seed cố định: tổng 200/100 ảnh. Chọn theo vị trí **bên trong split train/validation đã có**, giữ nguyên transforms của từng tập và lưu indices gốc để kiểm tra lại.

Cả hai model sẽ dùng đúng các ảnh này. Cell tạo lại DataLoader cho từng model để bắt đầu từ cùng seed shuffle. Đây là smoke test, chưa phải protocol so sánh chất lượng cuối cùng.

In [3]:
# Xác nhận dataset đã tải trước khi gọi loader của project.
for is_train in (True, False):
    raw_dataset = CIFAR10(smoke_config['data_root'], train=is_train, download=False)
    del raw_dataset
full_train, full_val, _unused_test, class_names = get_dataloaders(smoke_config)
assert len(class_names) == 10
saved_split = json.loads(Path(smoke_config['split_path']).read_text(encoding='utf-8'))
assert saved_split['seed'] == SEED and saved_split['val_size'] == smoke_config['val_size']

def choose_balanced_subset(source_subset, per_class, seed):
    labels = np.asarray(source_subset.dataset.targets)[source_subset.indices]
    rng = np.random.default_rng(seed)
    positions = []
    for class_id in range(len(class_names)):
        candidates = np.flatnonzero(labels == class_id)
        assert len(candidates) >= per_class
        positions.extend(rng.choice(candidates, per_class, replace=False).tolist())
    positions = sorted(positions)
    original_indices = [int(source_subset.indices[i]) for i in positions]
    return Subset(source_subset, positions), original_indices

tiny_train, tiny_train_indices = choose_balanced_subset(full_train.dataset, 20, SEED)
tiny_val, tiny_val_indices = choose_balanced_subset(full_val.dataset, 10, SEED + 1)
assert len(tiny_train) == 200 and len(tiny_val) == 100
assert not (set(tiny_train_indices) & set(tiny_val_indices)), 'Train/validation bị trùng ảnh.'
assert set(tiny_train_indices) <= set(saved_split['train_indices'])
assert set(tiny_val_indices) <= set(saved_split['val_indices'])

manifest_path = check_dir / 'subset_indices.json'
manifest = {'run_group': RUN_GROUP, 'split_path': smoke_config['split_path'],
            'split_seed': SEED, 'train_sampling_seed': SEED, 'val_sampling_seed': SEED + 1,
            'class_names': list(class_names), 'train_original_indices': tiny_train_indices,
            'val_original_indices': tiny_val_indices, 'train_samples': 200, 'val_samples': 100}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
smoke_config.update({'subset_manifest': manifest_path.as_posix(),
                     'subset_train_samples': 200, 'subset_val_samples': 100})

def make_smoke_loaders():
    generator = torch.Generator().manual_seed(SEED)
    train = DataLoader(tiny_train, batch_size=smoke_config['batch_size'], shuffle=True,
                       num_workers=smoke_config['num_workers'], pin_memory=DEVICE.type == 'cuda',
                       generator=generator)
    val = DataLoader(tiny_val, batch_size=smoke_config['batch_size'], shuffle=False,
                     num_workers=smoke_config['num_workers'], pin_memory=DEVICE.type == 'cuda')
    return train, val

checks['subset'] = True
print('PASS — subset: 200 train / 100 validation, 10 lớp cân bằng, không trùng ảnh.')
print('Indices được lưu tại:', manifest_path)

PASS — subset: 200 train / 100 validation, 10 lớp cân bằng, không trùng ảnh.
Indices được lưu tại: artifacts\notebook_smoke\cifar_subset\checks\20261109_005858_994753\subset_indices.json


## B2. Hàm chạy một epoch bằng training loop của project

Mỗi model được khởi tạo pretrained mới. Dùng cùng `smoke_config` cho DataLoader và `train_model()`, nhờ vậy input/batch/worker trong logs khớp dữ liệu thật.

Cell xác nhận có một dòng metrics, loss/accuracy/thời gian hữu hạn và đủ checkpoint/config/summary. AMP được training loop bật khi config cho phép và thiết bị là CUDA.

In [4]:
def train_one_smoke_model(model_name):
    config = dict(smoke_config)
    config['model'] = model_name
    config['run_name'] = model_name + '_' + RUN_GROUP
    train_loader, val_loader = make_smoke_loaders()
    print('Bắt đầu:', model_name, '| 1 epoch | 200 train / 100 validation', flush=True)
    summary = train_model(model_name, config, train_loader, val_loader, class_names, device=DEVICE)
    assert summary['epochs_completed'] == 1
    for key in ('best_checkpoint', 'last_checkpoint', 'metrics_csv'):
        assert Path(summary[key]).is_file(), f'Thiếu artifact: {key}'
    log_dir = Path(summary['metrics_csv']).parent
    assert (log_dir / 'config.json').is_file() and (log_dir / 'summary.json').is_file()
    with Path(summary['metrics_csv']).open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == 1
    for key in ('train_loss', 'val_loss', 'train_accuracy', 'val_accuracy', 'epoch_seconds'):
        assert math.isfinite(float(rows[0][key])), f'Metric không hữu hạn: {key}'
    assert 0 <= float(rows[0]['train_accuracy']) <= 1
    assert 0 <= float(rows[0]['val_accuracy']) <= 1
    resolved = json.loads((log_dir / 'config.json').read_text(encoding='utf-8'))
    for key in ('input_size', 'batch_size', 'num_workers', 'subset_manifest'):
        assert resolved[key] == config[key], f'Config log bị lệch: {key}'
    print('PASS — training smoke:', model_name)
    print('Checkpoint:', summary['best_checkpoint'])
    print('Validation loss/accuracy smoke:', summary['best_val_loss'], summary['best_val_accuracy'])
    return summary

## B3. Chạy ResNet18 pretrained

Cell này thực sự huấn luyện một epoch trên tập con 200 ảnh. Accuracy chỉ là chỉ báo trong phép thử ngắn.

In [5]:
checks['resnet_training'] = False
training_results['resnet18'] = train_one_smoke_model('resnet18')
checks['resnet_training'] = True
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

Bắt đầu: resnet18 | 1 epoch | 200 train / 100 validation


INFO | Model=resnet18 device=cuda AMP=True
INFO | Parameters total=11181642 trainable=11181642
INFO | Epoch 1/1 train loss=2.3323 acc=0.1550 val loss=2.1467 acc=0.2600 time=8.4s


PASS — training smoke: resnet18
Checkpoint: artifacts\notebook_smoke\cifar_subset\checkpoints\resnet18_20261109_005858_994753\best.pt
Validation loss/accuracy smoke: 2.146650390625 0.26


## B4. Chạy ConvNeXt-Tiny pretrained

Dùng cùng tập con đã lưu và cùng helper tạo DataLoader. Không lấy checkpoint ResNet18 làm đầu vào cho ConvNeXt; hai kiến trúc được khởi tạo và train riêng.

In [6]:
checks['convnext_training'] = False
training_results['convnext_tiny'] = train_one_smoke_model('convnext_tiny')
checks['convnext_training'] = True
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

Bắt đầu: convnext_tiny | 1 epoch | 200 train / 100 validation


INFO | Model=convnext_tiny device=cuda AMP=True
INFO | Parameters total=27827818 trainable=27827818
INFO | Epoch 1/1 train loss=2.0906 acc=0.3550 val loss=1.6513 acc=0.5700 time=21.6s


PASS — training smoke: convnext_tiny
Checkpoint: artifacts\notebook_smoke\cifar_subset\checkpoints\convnext_tiny_20261109_005858_994753\best.pt
Validation loss/accuracy smoke: 1.651279296875 0.57


## C1. Cách nạp checkpoint và kiểm tra dự đoán

Checkpoint project lưu **state_dict + metadata**, không lưu nguyên model. Khi khôi phục:

1. Đọc checkpoint về CPU, xác nhận model name, số lớp, class order, config và trạng thái optimizer/scheduler.
2. Dựng đúng kiến trúc với `pretrained=False`, rồi `load_state_dict(..., strict=True)`. Không cần tải lại ImageNet vì checkpoint đã chứa trọng số.
3. Chuyển model sang thiết bị, gọi `eval()`, tính lại validation với cùng chế độ AMP và đối chiếu metrics lúc lưu.
4. Với một batch cố định, lấy logits tham chiếu; lưu state_dict vào bộ đệm RAM, dựng model mới và nạp lại. So sánh logits trong cùng chế độ float32 bằng `torch.testing.assert_close()`.

Metadata hiện có `TorchVersion`, nên helper dùng `weights_only=True` và chỉ cho phép kiểu này trong ngữ cảnh đọc checkpoint do project vừa tạo. Tham khảo [torch.load](https://docs.pytorch.org/docs/2.14/generated/torch.load.html) và [serialization/safe_globals](https://docs.pytorch.org/docs/2.14/notes/serialization.html).

In [7]:
from torch.torch_version import TorchVersion

def read_project_checkpoint(path):
    with torch.serialization.safe_globals([TorchVersion]):
        return torch.load(path, map_location='cpu', weights_only=True)

def check_checkpoint_reload(model_name):
    summary = training_results[model_name]
    checkpoint = read_project_checkpoint(summary['best_checkpoint'])
    assert checkpoint['model_name'] == model_name
    assert checkpoint['num_classes'] == len(class_names)
    assert checkpoint['class_names'] == list(class_names)
    assert checkpoint['epoch'] == summary['best_epoch'] == 1
    assert checkpoint['optimizer_state_dict']['state'], 'Optimizer chưa có trạng thái cập nhật.'
    assert checkpoint['scheduler_state_dict'] is not None
    for key in ('input_size', 'batch_size', 'num_workers', 'subset_manifest'):
        assert checkpoint['config'][key] == smoke_config[key], f'Checkpoint config lệch: {key}'
    assert checkpoint['config']['pretrained'] is True

    model = build_model(model_name, num_classes=checkpoint['num_classes'],
                        pretrained=False, freeze_backbone=False)
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    model = model.to(DEVICE).eval()
    _, val_loader = make_smoke_loaders()
    amp_enabled = bool(checkpoint['config']['amp_enabled'])
    measured = run_epoch(model, val_loader, DEVICE, amp_enabled=amp_enabled)
    assert measured['samples'] == len(tiny_val)
    assert math.isclose(measured['accuracy'], summary['best_val_accuracy'], abs_tol=1e-8)
    assert math.isclose(measured['loss'], summary['best_val_loss'], rel_tol=1e-3, abs_tol=1e-4)

    reference_images, reference_labels = next(iter(val_loader))
    reference_images = reference_images.to(DEVICE)
    with torch.inference_mode():
        reference_logits = model(reference_images).detach().cpu()
    assert reference_logits.shape == (len(reference_labels), len(class_names))
    assert torch.isfinite(reference_logits).all().item()

    with io.BytesIO() as buffer:
        torch.save(model.state_dict(), buffer)
        buffer.seek(0)
        reloaded = build_model(model_name, num_classes=len(class_names), pretrained=False)
        reloaded.load_state_dict(torch.load(buffer, map_location='cpu', weights_only=True), strict=True)
    reloaded = reloaded.to(DEVICE).eval()
    with torch.inference_mode():
        actual_logits = reloaded(reference_images).detach().cpu()
    torch.testing.assert_close(actual_logits, reference_logits, rtol=1e-5, atol=1e-6)
    assert torch.equal(actual_logits.argmax(1), reference_logits.argmax(1))
    result = {'model': model_name, 'checkpoint': summary['best_checkpoint'],
              'validation_loss': measured['loss'], 'validation_accuracy': measured['accuracy'],
              'roundtrip_max_abs_difference': (actual_logits - reference_logits).abs().max().item(),
              'metadata_match': True, 'strict_load_passed': True, 'roundtrip_passed': True}
    print('PASS — checkpoint reload:', model_name)
    print('Validation khớp log; logits trước/sau lưu–nạp khớp. Max abs diff:', result['roundtrip_max_abs_difference'])
    return result

## C2. Kiểm tra checkpoint của cả hai model

Nếu có lỗi về tên lớp, kiến trúc, cấu hình hoặc trọng số, `assert`/`load_state_dict` sẽ báo tại cell này. So sánh logits dùng đúng một batch và cùng chế độ eval/float32, tách với việc tính validation bằng AMP theo training.

In [8]:
checks['checkpoint_reload'] = False
for model_name in ('resnet18', 'convnext_tiny'):
    reload_results[model_name] = check_checkpoint_reload(model_name)
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
checks['checkpoint_reload'] = True

PASS — checkpoint reload: resnet18
Validation khớp log; logits trước/sau lưu–nạp khớp. Max abs diff: 0.0
PASS — checkpoint reload: convnext_tiny
Validation khớp log; logits trước/sau lưu–nạp khớp. Max abs diff: 0.0


## 4. Tổng hợp và lưu bằng chứng

Kết quả cuối cần có ba dòng PASS tương ứng A, B, C. `verification_results.json` lưu đường dẫn checkpoint, indices tập con, metrics và kết quả so sánh logits.

Sau khi đọc kết quả, Phúc có thể cập nhật M1/T1/T2/T3 trong checklist, chia sẻ artifacts/interface checkpoint cho Quy và thống nhất protocol đầy đủ với Khang/Quy. Phép thử này chưa chứng minh model đã hội tụ, chưa kiểm tra resume training và chưa phải kết quả báo cáo CIFAR-10 cuối cùng.

In [9]:
required = ('backward', 'subset', 'resnet_training', 'convnext_training', 'checkpoint_reload')
assert all(checks.get(name, False) for name in required), f'Còn bước chưa đạt: {checks}'
verification = {'run_group': RUN_GROUP, 'device': str(DEVICE),
                'torch_version': str(torch.__version__),
                'scope': '1 update FP32; 1 epoch/model on 200 train + 100 validation; checkpoint reload',
                'subset_manifest': manifest_path.as_posix(), 'checks': checks,
                'backward': backward_result, 'training': training_results, 'reload': reload_results}
report_path = check_dir / 'verification_results.json'
report_path.write_text(json.dumps(verification, ensure_ascii=False, indent=2), encoding='utf-8')
print('PASS A — ConvNeXt-Tiny backward và cập nhật backbone/classifier.')
print('PASS B — Hai pretrained model train/validation một epoch trên CIFAR-10 nhỏ.')
print('PASS C — Checkpoint metadata, validation và logits lưu–nạp khớp.')
print('Kết quả kiểm chứng:', report_path)
print('Đây là smoke test, không dùng accuracy này để kết luận so sánh hai model.')

PASS A — ConvNeXt-Tiny backward và cập nhật backbone/classifier.
PASS B — Hai pretrained model train/validation một epoch trên CIFAR-10 nhỏ.
PASS C — Checkpoint metadata, validation và logits lưu–nạp khớp.
Kết quả kiểm chứng: artifacts\notebook_smoke\cifar_subset\checks\20261109_005858_994753\verification_results.json
Đây là smoke test, không dùng accuracy này để kết luận so sánh hai model.
